<a href="https://colab.research.google.com/github/Blandalytics/3D_wOBA/blob/main/3d_wOBA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

MLB Blog on xwOBA

https://technology.mlblogs.com/an-introduction-to-expected-weighted-on-base-average-xwoba-29d6070ba52b

# Import and Load

In [ ]:
from google.colab import drive
drive.mount('/gdrive')

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

from xgboost import XGBClassifier

# Yay Google integrations
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe
creds, _ = default()
gc = gspread.authorize(creds)

In [ ]:
%cd /gdrive/MyDrive/PL_Stats

In [ ]:
# Load trained/fitted models

# Right-handed KNN
with open('3D wOBA/3d_wOBA_R_knn.pkl', 'rb') as f:
    knn_clf_R = pickle.load(f)

# Left-Handed KNN
with open('3D wOBA/3d_wOBA_L_knn.pkl', 'rb') as f:
    knn_clf_L = pickle.load(f)

# XGBoost
with open('3D wOBA/3d_wOBA_xgb.pkl', 'rb') as f:
    xgb_clf = pickle.load(f)

In [ ]:
# Target Google Sheet
xwOBA_sheet = gc.open('3D wOBAcon')

In [ ]:
# Load Batter Names from MLB ID
name_df = pd.read_csv('https://docs.google.com/spreadsheets/d/1JgczhD5VDQ1EiXqVG-blttZcVwbZd5_Ne_mefUGwJnk/export?format=csv&gid=0')
name_df = name_df[['MLBID','MLBNAME']].dropna().astype({'MLBID':'int'}).set_index('MLBID').to_dict()['MLBNAME']

In [ ]:
# Load previously scraped data (credit: pybaseball's scraper!)
pitch_data = pd.concat([pd.read_parquet('PLV/Pitch Data (2017-2021).parquet.gzip'),
                        pd.read_parquet('PLV/Pitch Data (2022).parquet.gzip')], 
                       ignore_index=True)
pitch_data.shape

## Generate Spray Angle

0 is Pulled along the foul line, 90 is down the Opposite field foul line

In [ ]:
# Generat Spray Angle
home_plate = {'x':125,'y':-200} #plate coords
pitch_data['hc_y'] = -pitch_data['hc_y'] # Have homeplate be at the bottom of the plot 

pitch_data['tan_value'] = None
pitch_data.loc[pitch_data['hc_y'].notna(),'tan_value'] = pitch_data.loc[pitch_data['hc_y'].notna(),'hc_y'].sub(home_plate['y']).div(abs(pitch_data.loc[pitch_data['hc_y'].notna(),'hc_x'].sub(home_plate['x'])))

pitch_data['spray_deg'] = None
pitch_data.loc[(pitch_data['hc_x']>=125) &
               (pitch_data['stand']=='L'),'spray_deg'] = pitch_data.loc[(pitch_data['hc_x']>=125) &
                                                                         (pitch_data['stand']=='L'),'tan_value'].apply(lambda x: math.degrees(math.atan(x))-45)
pitch_data.loc[(pitch_data['hc_x']<125) &
               (pitch_data['stand']=='L'),'spray_deg'] = pitch_data.loc[(pitch_data['hc_x']<125) &
                                                                         (pitch_data['stand']=='L'),'tan_value'].apply(lambda x: 135-math.degrees(math.atan(x)))

pitch_data.loc[(pitch_data['hc_x']>=125) &
               (pitch_data['stand']=='R'),'spray_deg'] = pitch_data.loc[(pitch_data['hc_x']>=125) &
                                                                         (pitch_data['stand']=='R'),'tan_value'].apply(lambda x: 135-math.degrees(math.atan(x)))
pitch_data.loc[(pitch_data['hc_x']<125) &
               (pitch_data['stand']=='R'),'spray_deg'] = pitch_data.loc[(pitch_data['hc_x']<125) &
                                                                         (pitch_data['stand']=='R'),'tan_value'].apply(lambda x: math.degrees(math.atan(x))-45)

pitch_data['hc_y'] = -pitch_data['hc_y'] # Revert to original Y coords 
pitch_data['spray_deg'] = pitch_data['spray_deg'].astype('float')

# Train Models

In [ ]:
# Model data
xwOBA_df = (
    pitch_data
    .loc[(pitch_data['game_type']=='R') & # Regular Season only
         pitch_data['spray_deg'].notna() & # Need Spray Angle
         pitch_data['launch_angle'].notna() & # Need Launch Angle
         pitch_data['launch_speed'].notna(), # Need Exit Velo
         ['stand','spray_deg', 'launch_angle','launch_speed', # Model columns
          'estimated_woba_using_speedangle','woba_value',  # Comparison columns
          'batter','game_year','events','bb_type']] # Other Columns
    .assign(Batter=lambda x: x.batter.map(name_df)) # Helpful to have names
    .rename(columns={'woba_value':'wOBA',
                     'estimated_woba_using_speedangle':'xwOBA',
                     'game_year':'Season'})
    .copy()
    )

xwOBA_df['stand_cat'] = 0
xwOBA_df.loc[xwOBA_df['stand']=='R','stand_cat'] = 1

xwOBA_df = xwOBA_df.astype({
    'launch_angle':'int',
    'launch_speed':'float'
})

# Remove autofilled speed/angle combinations
# https://tht.fangraphs.com/43416-2/ 
xwOBA_df['no_null_mask'] = 0
xwOBA_df.loc[(xwOBA_df['bb_type']=='popup') &
             (xwOBA_df['launch_speed']==80) &
             (xwOBA_df['launch_angle']==69),'no_null_mask'] = 1
xwOBA_df.loc[(xwOBA_df['bb_type']=='ground_ball') &
             (xwOBA_df['launch_speed']==82.9) &
             (xwOBA_df['launch_angle']==-21),'no_null_mask'] = 1

In [ ]:
wOBA_cats = {
    0:0,
    0.9:1,
    1.25:2,
    1.6:3,
    2:4
}

target = 'woba_value_cat'
xwOBA_df[target] = xwOBA_df['wOBA'].map(wOBA_cats).astype(int)

# Previously tuned hyperparams
parameters_knn = {
    'n_neighbors': 100,
    'leaf_size': 20,
    'p': 1,
    'weights': 'uniform',
    'metric': 'minkowski'
}

parameters_xgb = {
    'n_estimators':400,
    'seed_num':12,
    'objective':'multi:softprob',
    'max_depth': 6, 
    'min_child_weight': 4,
    'gamma':0,
    'colsample_bytree': 0.9, 
    'subsample': 0.9,
    'alpha':10,
    'learning_rate':0.2
}

## KNN Classifier

In [ ]:
knn_clf = KNeighborsClassifier(**parameters_knn)

for handedness in ['R','L']:
  temp_df = xwOBA_df.loc[(xwOBA_df['stand']==handedness) &
                         (xwOBA_df['no_null_mask']==0)].copy()
  knn_clf.fit(temp_df[['spray_deg','launch_angle','launch_speed']],
              temp_df[target])
  # Save Model
  #with open('3D wOBA/3d_wOBA_{}_knn.pkl'.format(handedness), 'wb') as f:
  #  pickle.dump(knn_clf, f)
  
  xwOBA_df.loc[(xwOBA_df['stand']==handedness),['out_knn','single_knn','double_knn','triple_knn','home_run_knn']] = knn_clf.predict_proba(xwOBA_df.loc[(xwOBA_df['stand']==handedness),['spray_deg','launch_angle','launch_speed']])

xwOBA_df['3d_wOBA_knn'] = xwOBA_df['single_knn']*0.9 + xwOBA_df['double_knn']*1.25 + xwOBA_df['triple_knn']*1.6 + xwOBA_df['home_run_knn']*2

In [ ]:
xwOBA_df[['out_knn','single_knn','double_knn','triple_knn','home_run_knn','wOBA','xwOBA','3d_wOBA_knn']].describe().round(4)

## XGBoost Classifier

In [ ]:
# xgboost goes brrrrr
X_train, X_test, y_train, y_test = train_test_split(
    xwOBA_df.loc[(xwOBA_df['no_null_mask']==0),
                 ['stand_cat','spray_deg','launch_angle','launch_speed']], 
    xwOBA_df.loc[(xwOBA_df['no_null_mask']==0),
                 target],
    test_size=0.25, 
    random_state = parameters_xgb['seed_num'])

xgb_clf = XGBClassifier(**parameters_xgb)

xgb_clf.fit(X_train,
            y_train)

# Save Model
#with open('3D wOBA/3d_wOBA_xgb.pkl', 'wb') as f:
#    pickle.dump(xgb_clf, f)

xwOBA_df[['out_xgb','single_xgb','double_xgb','triple_xgb','home_run_xgb']] = xgb_clf.predict_proba(xwOBA_df[['stand_cat','spray_deg','launch_angle','launch_speed']])

xwOBA_df['3d_wOBA_xgb'] = xwOBA_df['single_xgb']*0.9 + xwOBA_df['double_xgb']*1.25 + xwOBA_df['triple_xgb']*1.6 + xwOBA_df['home_run_xgb']*2

In [ ]:
xwOBA_df[['out_xgb','single_xgb','double_xgb','triple_xgb','home_run_xgb','wOBA','xwOBA','3d_wOBA_xgb']].describe().round(4)

## Group Seasons

In [ ]:
batter_df = (
    xwOBA_df
    .groupby(['Season',
              'Batter'],as_index=False)
    [['wOBA','xwOBA','3d_wOBA_knn','3d_wOBA_xgb','launch_angle']]
    .agg({
        'wOBA':'mean',
        'xwOBA':'mean',
        '3d_wOBA_knn':'mean',
        '3d_wOBA_xgb':'mean',
        'launch_angle':'count'
    })
    .rename(columns={'launch_angle':'BBE'})
    .sort_values(['Batter','Season'])
    .reset_index(drop=True)
    .copy()
)

In [ ]:
# Add next year's values, for checking predictiveness
for stat in ['BBE','wOBA']:
  batter_df[stat+'_y+1'] = batter_df[stat].shift(-1)
  batter_df.loc[(batter_df['Batter']!=batter_df['Batter'].shift(-1)) |
                ((batter_df['Season']+1)!=batter_df['Season'].shift(-1)),stat+'_y+1'] = None

# Ensemble weighting
find best combination of KNN model and XGBoost model that minimizes the quadratic mean of current & next year wOBA errors

In [ ]:
def model_ensemble(comp_stat,comp_stat_2,model_1_pred,model_2_pred,df=batter_df):
  print('Predicting '+str(comp_stat)+' & '+str(comp_stat_2))
  temp_df = df[[comp_stat,comp_stat_2,model_1_pred,model_2_pred]].dropna().copy()
  
  best_error = np.infty
  best_weight = 0
  for weight in np.linspace(0,1,1001): # for every 0.1% weight, compare error
    temp_df['ensemble_pred'] = weight*temp_df[model_1_pred] + (1-weight)*temp_df[model_2_pred]
    error_1 = temp_df['ensemble_pred'].sub(temp_df[comp_stat]).abs().mean()
    error_2 = temp_df['ensemble_pred'].sub(temp_df[comp_stat_2]).abs().mean()
    error = (error_1**2+error_2**2)**0.5

    if error < best_error:
      best_error = error
      best_weight = weight
    if weight==0:
      print(model_2_pred+' Error: {}'.format(round(error,4)))
    if weight==1:
      print(model_1_pred+' Error: {}'.format(round(error,4)))
  print('Best Error: {}'.format(round(best_error,4)))
  print('Best Weight: {}'.format(best_weight))
  return best_weight

In [ ]:
ensemble_weight = model_ensemble('wOBA','wOBA_y+1',
                                 model_1_pred='3d_wOBA_knn',
                                 model_2_pred='3d_wOBA_xgb')

# Apply Models

## KNN

In [ ]:
def add_3d_wOBA_knn(df,models=[knn_clf_R,knn_clf_L]):
  for handedness in ['R','L']:
    knn_clf = models[0] if handedness=='R' else models[1]
    df.loc[df['stand']==handedness,['out_knn','single_knn','double_knn','triple_knn','home_run_knn']] = knn_clf.predict_proba(df.loc[df['stand']==handedness,['spray_deg','launch_angle','launch_speed']])
  return df['single_knn']*0.9 + df['double_knn']*1.25 + df['triple_knn']*1.6 + df['home_run_knn']*2

xwOBA_df['3d_wOBA_knn'] = add_3d_wOBA_knn(xwOBA_df)

In [ ]:
# KNN Distribution
sns.kdeplot(x=xwOBA_df['3d_wOBA_knn'])
sns.despine()

## XGB

In [ ]:
def add_3d_wOBA_xgb(df,model=xgb_clf):
  df['stand_cat'] = 0
  df.loc[df['stand']=='R','stand_cat'] = 1

  df = df.astype({
      'launch_angle':'int',
      'launch_speed':'float'
  })

  df[['out_xgb','single_xgb','double_xgb','triple_xgb','home_run_xgb']] = model.predict_proba(df[['stand_cat','spray_deg','launch_angle','launch_speed']])
  return df['single_xgb']*0.9 + df['double_xgb']*1.25 + df['triple_xgb']*1.6 + df['home_run_xgb']*2

xwOBA_df['3d_wOBA_xgb'] = add_3d_wOBA_xgb(xwOBA_df)

In [ ]:
# XGB Distribution
sns.kdeplot(x=xwOBA_df['3d_wOBA_xgb'])
sns.despine()

## Combined

In [ ]:
def combine_3d_wOBA(df,model_weight=ensemble_weight):
  df['3D wOBA'] = df['wOBA'].copy() # Baseline is wOBA (non-BBE values)
  df.loc[df['3d_wOBA_knn'].notna() &
         df['3d_wOBA_xgb'].notna(),'3D wOBA'] = model_weight * df.loc[df['3d_wOBA_knn'].notna() &
                                                                      df['3d_wOBA_xgb'].notna(),'3d_wOBA_knn'] + (1-model_weight) * df.loc[df['3d_wOBA_knn'].notna() &
                                                                                                                                           df['3d_wOBA_xgb'].notna(),'3d_wOBA_xgb']

  return df['3D wOBA']
xwOBA_df['3D wOBA'] = combine_3d_wOBA(xwOBA_df)

In [ ]:
# 3D wOBA distribution
sns.kdeplot(x=xwOBA_df['3D wOBA'])
sns.despine()

In [ ]:
# Fill non_BBE xwOBA with wOBA
xwOBA_df['xwOBA'] = xwOBA_df['xwOBA'].fillna(xwOBA_df['wOBA'])

# Analyze Results

In [ ]:
batter_df = (
    xwOBA_df
    .groupby(['Season',
              'Batter'],as_index=False)
    [['wOBA','xwOBA','3d_wOBA_knn','3d_wOBA_xgb','3D wOBA','launch_angle']]
    .agg({
        'wOBA':'mean',
        'xwOBA':'mean',
        '3d_wOBA_knn':'mean',
        '3d_wOBA_xgb':'mean',
        '3D wOBA':'mean',
        'launch_angle':'count'
    })
    .rename(columns={'launch_angle':'BBE'})
    .sort_values(['Batter','Season'])
    .reset_index(drop=True)
    .sort_values('3D wOBA', ascending=False)
    .query('BBE>=200') # BBE min
    .copy()
)

batter_df['Expectation Diff'] = batter_df['xwOBA'].sub(batter_df['3D wOBA']) # comparison of expected stats
batter_df['3D Diff'] = batter_df['wOBA'].sub(batter_df['3D wOBA']) # comparison to actual results

# Update Google Sheet
#set_with_dataframe(xwOBA_sheet.get_worksheet(0),
#                   batter_df[['Batter','Season','BBE','wOBA','xwOBA','3D wOBA',
#                              'Expectation Diff','3D Diff']],
#                   resize=True)

In [ ]:
print('Best 3D wOBAcon season')
batter_df.loc[batter_df['3D wOBA']==batter_df['3D wOBA'].max()]

In [ ]:
print('Worst 3D wOBAcon season')
batter_df.loc[batter_df['3D wOBA']==batter_df['3D wOBA'].min()]

In [ ]:
# Individual Batter comparison
batter_df.loc[batter_df['Batter']=='Alex Bregman'].drop(columns=['Expectation Diff','3D Diff']).sort_values('Season').round(3)

In [ ]:
print('3D wOBAcon Overperformers')
(batter_df
 .sort_values('3D Diff', ascending=False)
 .drop(columns=['Expectation Diff'])
 .head(5)
 .round(3)
 )

In [ ]:
print('3D wOBAcon Underperformers')
(batter_df
 .sort_values('3D Diff')
 .drop(columns=['Expectation Diff'])
 .head(5)
 .round(3)
 )

In [ ]:
# Residual comparison
sns.kdeplot(x=batter_df['3D Diff'],
            cut=0)
sns.despine()

In [ ]:
# Event Comparisons
(xwOBA_df
 .loc[xwOBA_df['events'].isin(
     ['double','double_play','field_error','field_out','fielders_choice',
      'fielders_choice_out','grounded_into_double_play','home_run','sac_bunt',
      'sac_bunt_double_play','sac_fly','sac_fly_double_play','single','triple',
      'triple_play'])] # Relevant events (rel-events?)
 .groupby('events')
 [['wOBA','xwOBA','3D wOBA']]
 .mean()
 .round(3)
 )

In [ ]:
# Model Distributions on BBE
fig, ax = plt.subplots(figsize=(8,4))
sns.kdeplot(x=xwOBA_df['xwOBA'])
sns.kdeplot(x=xwOBA_df['3d_wOBA_knn'])
sns.kdeplot(x=xwOBA_df['3d_wOBA_xgb'])
sns.kdeplot(x=xwOBA_df['3D wOBA'])
ax.set(xlabel='wOBA Scale')
ax.legend(['xwOBA','3d_wOBA_knn','3d_wOBA_xgb','3D wOBA'])
sns.despine()

In [ ]:
# Best batters (2017-2022)
all_seasons_df = (
    xwOBA_df
    .groupby('Batter',as_index=False)
    [['wOBA','xwOBA','3D wOBA','launch_angle']]
    .agg({
        'wOBA':'mean',
        'xwOBA':'mean',
        '3D wOBA':'mean',
        'launch_angle':'count'
    })
    .rename(columns={'launch_angle':'BBE'})
    .query('BBE >=500')
    .sort_values('3D wOBA', ascending=False)
    [['Batter','BBE','wOBA','xwOBA','3D wOBA']]
)
all_seasons_df['3D Diff'] = all_seasons_df['wOBA'].sub(all_seasons_df['3D wOBA'])
#set_with_dataframe(xwOBA_sheet.get_worksheet(1),all_seasons_df, resize=True)

In [ ]:
print('Best 3D wOBAcon batters (2017-2022)')
all_seasons_df.head(10).round(3)